In [ ]:
import os
from typing import List, Dict
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from pathlib import Path

def str_getfrom_token_first(s1: str, tok: str):
    if not tok: return str
    pos = s1.find(tok)
    return s1[pos:] if pos >= 0 else s1
    
class BibleHubHTMLParser:
    def __init__(self):
        pass
    
    def parse_html_folder_biblehub(self, folder_path: str, book_filter: List[str]= None, save_result=False) -> Dict[str, List[List[str]]]:
        """Parse HTML files from BibleHub"""
        result = {}
        sentence_tokens = []
        df_result = []
        iterator_folders = []
        if book_filter is None:
            iterator_folders = os.listdir(folder_path)
        else:
            #if any(book.lower() in folder.lower() for book in book_filter)
            #dircont =
            iterator_folders = [n.lower() for n in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, n))
                       and n.lower() in book_filter]
            #for book in book_filter:
            #    if book.lower()  in dircont: iterator_folders.append(book)
        urls_greek = {}
        chdone = 0
        totchs = len(iterator_folders)
        for fold in iterator_folders:
            chapsnames = [filename for filename in os.listdir(folder_path.strip(os.path.sep)+os.path.sep+fold) 
                          if filename.endswith('.htm') or filename.endswith('.html')]
            ccchdone = 0
            totchhh = len(chapsnames)
            for filename in chapsnames:
                if(True):
                    chapter = os.path.splitext(filename)[0]
                    current_book = fold
                    result[f"{current_book}_{chapter}"] = []
                    print(f"{current_book} {chapter} ( {float(chdone)/float(totchs)*float(100)+(float(1)/float(totchs))*(float(ccchdone)/float(totchhh))*float(100):.1f}% )")
                    soup = BeautifulSoup(open(os.path.join(folder_path, fold, filename)), 'html.parser')
                    
                    table_spans = soup.find_all('table', class_='tablefloat')
                    curr_verse = 1
                    strong_num=-1
                    strong_title = ''
                    strong_en_title = ''
                    translit=''
                    translit_title=''
                    form = ''
                    form_en = ''
                    curr_node = None
                    word_idx = 0
                    for tbc in range(len(table_spans)):
                        table = table_spans[tbc]
                        
                        # 1. Verse loģika paliek tā pati
                        curr_node = table.find("span", class_='reftop')
                        if curr_node is not None:
                            curr_verse = int(curr_node.text.strip("\xa0").split(":")[-1])
                            word_idx = 0
                    
                        # 2. EKSTRAKCIJA: Dabūjam visus elementus kā sarakstus
                        # Grieķu vārdi (sadalām pa nbsp;)
                        greek_span = table.find("span", class_='greek')
                        greek_words = greek_span.text.split('\xa0') if greek_span else []
                        
                        # Angļu vārdi
                        eng_span = table.find("span", class_='eng')
                        eng_words = eng_span.text.split('\xa0') if eng_span else []
                    
                        # Strong numuri (savācam visus <a> tagus)
                        strong_links = []
                        strongs_container = table.find("span", class_='strongs') or table.find("span", class_='pos')
                        if strongs_container:
                            strong_links = strongs_container.find_all('a')
                    
                        # Translit (bieži vien ir tikai viens uz grupu, tāpēc uzmanīgi)
                        #translit_span = table.find("span", class_='translit')
                        #translit_val = translit_span.text.replace("\xa0", ' ') if translit_span else ""
                    
                        # 3. TRANSFORMĀCIJA: Ejam cauri visiem vārdiem šajā šūnā
                        # Izmantojam max garumu, lai nepazaudētu datus, ja saraksti nav vienādi
                        #num_words = max(len(greek_words), len(strong_links), len(eng_words))

                        s_nums = []
                        s_titles = []
                        for i in range(len(strong_links)):
                                # Notīrām numuru no domuzīmēm un liekiem simboliem
                            s_num_raw = strong_links[i].text.replace("-", "").strip()
                            s_nums.append(s_num_raw)
                            #if s_num_raw.isnumeric():
                            #    s_num = int(s_num_raw)
                            s_titles.append(strong_links[i].attrs.get('title', '').strip())
                            
                            # Pievienojam URL kolekcijai lejupielādei
                            attrb = strong_links[i].attrs.get('href', '')
                            if attrb.startswith('/greek'):
                                urls_greek[attrb] = f"curl -O https://www.biblehub.com{attrb}"
                    
                        xdata = {
                                'verse': curr_verse,
                                'word_idx': word_idx,
                                'form': " ".join(greek_words).strip(),
                                'form_en': " ".join(eng_words).strip(),
                                'strong_num': "|".join(s_nums),
                                'strong_title': "|".join(s_titles).strip(),
                                #'translit': translit_val if i == 0 else "", # Translit parasti liekam pie pirmā vārda
                                'book': fold,
                                'chapter': chapter
                        }
                    
                            # Pievienojam rezultātiem
                        if len(result[f"{current_book}_{chapter}"]) < curr_verse:
                            for _ in range(len(result[f"{current_book}_{chapter}"]), curr_verse):
                                result[f"{current_book}_{chapter}"].append([])
                        
                        result[f"{current_book}_{chapter}"][curr_verse-1].append(xdata)
                        df_result.append(xdata)
                        word_idx += 1
                #return result
                ccchdone +=1
            chdone += 1

        if(save_result):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{timestamp}_LXX_EXT_dload.sh"
            if(len(urls_greek.items()) > 0):
                with open(filename, "w") as file:
                    for k, v in urls_greek.items():
                        file.write(f"{v}\n")
            dbname = f"{timestamp}_biblehub_LXX_EXT_el_en_direct.csv"
                # Create DataFrame and write to CSV
            if(len(df_result)>0):
                df = pd.DataFrame(df_result)
                
                # Ensure output directory exists
                Path(dbname).parent.mkdir(parents=True, exist_ok=True)
                
                # Write to CSV with proper formatting
                df.to_csv(
                    dbname,
                    index=False,
                    encoding='utf-8',
                    quoting=1, 
                    escapechar='\\'
                )                    
        return result

In [ ]:
parser_html = BibleHubHTMLParser()
html_sources = parser_html.parse_html_folder_biblehub('_scrap_gk_nums/', None, True)